## Download Data

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/customers-100.csv https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/customers-100.csv



## CSV embeddings Example

In [ ]:
import pandas as pd
d = pd.read_csv("data/customers-100.csv")
d.columns

In [3]:
csv_path = "data/customers-100.csv"

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import chromadb

df = pd.read_csv(csv_path)
columns = ['Index', 'Customer Id', 'First Name', 'Last Name', 'Company', 'City',
       'Country', 'Phone 1', 'Phone 2', 'Email', 'Subscription Date',
       'Website']

model = SentenceTransformer('all-MiniLM-L6-v2')

client = chromadb.PersistentClient(path="chroma_db")
collection = client.get_or_create_collection("csv_embeddings")

for idx, row in df.iterrows():
    # Create document text
    text = ' | '.join(str(row[col]) for col in columns)
    
    # Create metadata dictionary
    metadata = {col: str(row[col]) for col in columns}
    
    # Generate embedding
    embedding = model.encode([text])[0].tolist()  # Single embedding
    
    collection.add(
        ids=[str(idx)],
        embeddings=[embedding],
        documents=[text],
        metadatas=[metadata]  # CRITICAL: Add metadata here
    )

print("Embeddings and metadata saved to ChromaDB.")


In [6]:
import os
from dotenv import load_dotenv
os.environ["GROQ_API_KEY"] = os.getenv('GROQ_API_KEY')  # Replace with your actual key


In [15]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Connect to your persisted ChromaDB
chroma_client = chromadb.PersistentClient(path="chroma_db")
chroma_collection = chroma_client.get_or_create_collection("csv_embeddings")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Set up embedding model (same as used for ingestion)
embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")

# Set up Groq LLM
llm = Groq(model="qwen-qwq-32b")  # or "llama3-70b-8192"


In [16]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)


In [ ]:
query_engine = index.as_query_engine(llm=llm, similarity_top_k=6)
response = query_engine.query("which company does sheryl Baxter work for?")
print(str(response))


In [ ]:
# Basic retrieval example using metadata filters 

from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.core import VectorStoreIndex

# Define the filter for country
filters = MetadataFilters(
    filters=[
        MetadataFilter(key="Country", value="United Arab Emirates", operator=FilterOperator.EQ)
    ]
)

# Create a retriever with the filter and a high similarity_top_k to ensure all matches are retrieved
retriever = index.as_retriever(filters=filters, similarity_top_k=16)

# Retrieve nodes (chunks) matching the filter
result_nodes = retriever.retrieve("List all customers from United Arab Emirates with their names.")

# Optionally, process results programmatically before passing to LLM
customers = []
for node in result_nodes:
    # Assuming you stored first/last name as metadata
    meta = node.metadata
    customers.append(f"{meta['First Name']} {meta['Last Name']}")

# If you want to print or use this list:
print(f"Found {len(customers)} customers from United Arab Emirates:")
print("\n".join(customers))


In [ ]:
print(chroma_collection.peek())  # Check stored metadata
# Print first entry's metadata
print(collection.get(ids=["0"], include=["metadatas"])["metadatas"][0])


## PDF embeddings Example

In [ ]:
import os
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb

# --- Step 1: Extract text from PDF ---
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

# --- Step 2: Split text into chunks (optional but recommended for long docs) ---
def split_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i+chunk_size])
        if chunk:
            chunks.append(chunk)
    return chunks

# --- Step 3: Generate embeddings ---
def embed_chunks(chunks, model_name="all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    embeddings = model.encode(chunks, show_progress_bar=True)
    return embeddings

# --- Step 4: Store embeddings in ChromaDB ---
def store_in_chromadb(chunks, embeddings, db_path="chroma_pdf_db", collection_name="pdf_embeddings"):
    client = chromadb.PersistentClient(path=db_path)
    collection = client.get_or_create_collection(collection_name)
    for idx, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        collection.add(
            ids=[str(idx)],
            embeddings=[embedding],
            documents=[chunk],
            metadatas=[{"chunk_id": idx}]
        )
    print(f"Stored {len(chunks)} chunks in ChromaDB.")

# --- Main pipeline ---
def pdf_to_vector_db(pdf_path):
    text = extract_text_from_pdf(pdf_path)
    chunks = split_text(text)
    embeddings = embed_chunks(chunks)
    store_in_chromadb(chunks, embeddings)

# --- Usage ---
pdf_path = "data/Understanding_Climate_Change.pdf"  # Replace with your PDF file path
pdf_to_vector_db(pdf_path)


In [3]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Connect to your persisted ChromaDB
chroma_client = chromadb.PersistentClient(path="chroma_pdf_db")
chroma_collection = chroma_client.get_or_create_collection("pdf_embeddings")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Set up embedding model (same as used for ingestion)
embed_model = HuggingFaceEmbedding(model_name="all-MiniLM-L6-v2")

# Set up Groq LLM
llm = Groq(model="qwen-qwq-32b")  


In [5]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)

In [ ]:
query_engine = index.as_query_engine(llm=llm, similarity_top_k=7)
response = query_engine.query("what is Droughts?")
print(str(response))
